# Preparación del Dataset de Pokemon

Este notebook se encarga de descargar, limpiar y preparar los datos para entrenar una red neuronal que clasifique Pokemon según su tipo principal.

### 1. Instalación de Dependencias
Instalamos la librería `kagglehub` para facilitar la descarga de los datasets desde Kaggle.

In [ ]:
!pip install -q kagglehub[pandas-datasets]

### 2. Importación de Librerías
Cargamos las librerías necesarias para el procesamiento de datos, manejo de imágenes y entrenamiento con PyTorch.

In [ ]:
import os
import json
import random
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import kagglehub
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

### 3. Configuración de Credenciales de Kaggle
Configuramos las credenciales necesarias para acceder a la API de Kaggle.

In [ ]:
KAGGLE_USERNAME = "tomsguiazu"
KAGGLE_KEY = "KGAT_bf485502d284e7837b119eb874b76139"

os.makedirs("/root/.kaggle", exist_ok=True)

kaggle_credentials = {
    "username": KAGGLE_USERNAME,
    "key": KAGGLE_KEY
}

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_credentials, f)

!chmod 600 /root/.kaggle/kaggle.json

print("Kaggle configurado correctamente")

### 4. Descarga y Extracción de Datasets
Descargamos los datasets de información tabular (CSV) y de imágenes de Pokemon.

In [ ]:
# Descargar dataset tabular
!curl -L -o pokemon.zip https://www.kaggle.com/api/v1/datasets/download/mlomuscio/pokemon
!unzip -q pokemon.zip -d pokemon_dataset

# Descargar dataset de imágenes
!curl -L -o pokemon_images.zip https://www.kaggle.com/api/v1/datasets/download/lantian773030/pokemonclassification
!unzip -q pokemon_images.zip -d pokemon_images_dataset

### 5. Carga y Limpieza de Datos Tabulares
Cargamos el CSV, filtramos por la Generación 1 y normalizamos los nombres para que coincidan con las carpetas de imágenes.

In [ ]:
CSV_PATH = "pokemon_dataset/PokemonData.csv"
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.lower()

# Filtrar solo Generación 1
df_gen1 = df[df["generation"] == 1][["name", "type1"]]

def normalize_name(name):
    return str(name).lower().replace(".", "").replace("'", "").replace("♀", "").replace("♂", "").replace(" ", "").replace("-", "")

df_gen1["normalized_name"] = df_gen1["name"].apply(normalize_name)

# Corregir nombres específicos y eliminar duplicados o errores
df_gen1["name"] = df_gen1["name"].replace({"Mr. Mime": "MrMime", "Farfetch'd": "Farfetchd"})
df_gen1 = df_gen1[~df_gen1["name"].isin(["Nidoranâ™€", "Nidoranâ™‚"])]
df_gen1 = df_gen1[~df_gen1["name"].str.contains("Mega", na=False)]

print(f"Cantidad de Pokemon luego de limpieza: {len(df_gen1)}")

### 6. Mapeo de Imágenes con Tipos
Recorremos las carpetas de imágenes y las asociamos con su tipo principal definido en el CSV.

In [ ]:
IMAGES_PATH = "pokemon_images_dataset/PokemonData"
image_data = []

for folder_name in os.listdir(IMAGES_PATH):
    folder_path = os.path.join(IMAGES_PATH, folder_name)
    if not os.path.isdir(folder_path): continue

    normalized_folder = normalize_name(folder_name)
    pokemon_match = df_gen1[df_gen1["normalized_name"] == normalized_folder]

    if pokemon_match.empty: continue

    pokemon_name = pokemon_match.iloc[0]["name"]
    pokemon_type = pokemon_match.iloc[0]["type1"]

    VALID_EXTENSIONS = (".jpg", ".jpeg", ".png")
    for image_name in os.listdir(folder_path):
        if not image_name.lower().endswith(VALID_EXTENSIONS): continue
        image_path = os.path.join(folder_path, image_name)
        image_data.append({"pokemon": pokemon_name, "type1": pokemon_type, "image_path": image_path})

final_df = pd.DataFrame(image_data)
print(f"Total de imágenes encontradas: {len(final_df)}")

### 7. Definición de Transformaciones e Índices de Clase
Preparamos las transformaciones (Data Augmentation para entrenamiento) y creamos el mapeo de tipos a índices numéricos.

In [ ]:
# Transformaciones
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Mapeo de clases
classes = sorted(final_df["type1"].unique())
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
idx_to_class = {v: k for k, v in class_to_idx.items()}

print("Mapeo de clases:", class_to_idx)

### 8. Clase Custom Dataset de PyTorch
Definimos una clase para cargar las imágenes y sus etiquetas de forma eficiente.

In [ ]:
class PokemonTypeDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = class_to_idx[row["type1"]]
        return image, label

### 9. División del Dataset
Dividimos los datos en conjuntos de Entrenamiento (70%), Validación (15%) y Prueba (15%) de forma estratificada por tipo.

In [ ]:
SEED = 43
train_df, temp_df = train_test_split(final_df, test_size=0.30, stratify=final_df["type1"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["type1"], random_state=SEED)

train_dataset = PokemonTypeDataset(train_df, transform=train_transform)
val_dataset = PokemonTypeDataset(val_df, transform=eval_transform)
test_dataset = PokemonTypeDataset(test_df, transform=eval_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

### 10. Visualización de Muestras
Mostramos algunas imágenes del dataset para verificar que las transformaciones y etiquetas sean correctas.

In [ ]:
def show_samples(dataset, title, num_samples=8):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))
    for i in range(num_samples):
        idx = random.randint(0, len(dataset) - 1)
        image, label = dataset[idx]
        image = image.permute(1, 2, 0).numpy() * std + mean
        image = np.clip(image, 0, 1)
        
        axes[i].imshow(image)
        axes[i].set_title(idx_to_class[label])
        axes[i].axis("off")
    plt.suptitle(title)
    plt.show()

show_samples(train_dataset, "Muestras de Entrenamiento")